# Comparative Analysis: Breast Cancer Diagnostic Classification

This notebook demonstrates a machine learning workflow for binary classification using the Breast Cancer Wisconsin (Diagnostic) dataset. The objective is to build and evaluate predictive models that categorize tumors as **Malignant (0)** or **Benign (1)**.

### Project Scope:
1.  **Data Ingestion & Pre-processing:** Implementing stratified sampling for robust evaluation.
2.  **Hyperparameter Optimization:** Utilizing `GridSearchCV` to optimize Decision Tree and Random Forest architectures.
3.  **Performance Evaluation:** Analyzing the variance-bias tradeoff through tree depth analysis and ensemble methods.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split


from sklearn.datasets import load_breast_cancer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score,precision_score,recall_score

from sklearn.metrics import confusion_matrix,classification_report

from sklearn.model_selection import GridSearchCV

## 1. Data Engineering and Partitioning

We utilize stratified splitting to ensure that the target class distribution is preserved across the training and testing subsets. This is critical in clinical datasets where class imbalance can lead to deceptive accuracy metrics.

In [3]:
df = load_breast_cancer(as_frame=True).frame
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [4]:
df.shape

(569, 31)

In [5]:
X = df.drop(['target'],axis=1)
y = df['target']
X_train , X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

## 2. Decision Tree Modeling & Parameter Tuning

We establish a baseline using a standard Decision Tree Classifier. To mitigate potential overfitting, we employ `GridSearchCV` to explore the parameter space, focusing on `criterion` (Gini vs. Entropy) and `max_depth` to determine the optimal pruning point.

In [6]:
model = DecisionTreeClassifier()

model.fit(X_train,y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90        42
           1       0.96      0.92      0.94        72

    accuracy                           0.92       114
   macro avg       0.91      0.92      0.92       114
weighted avg       0.92      0.92      0.92       114



In [7]:
grid = {
    "criterion" : ['gini','entropy'],
    "max_depth" :[1,2,3,4,5,6,7,8,None],
}

grid_search = GridSearchCV(
    estimator = model,
    param_grid = grid,
    cv = 5
)

grid_search.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, None]})

In [8]:
y_pred = grid_search.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.80      0.93      0.86        42
           1       0.95      0.86      0.91        72

    accuracy                           0.89       114
   macro avg       0.87      0.89      0.88       114
weighted avg       0.90      0.89      0.89       114



# Training on different depth

### Depth Sensitivity Analysis

The following cells examine the model's sensitivity to `max_depth`. By tracking the divergence between training and testing accuracy, we can identify the threshold where the model transitions from underfitting to overfitting (high variance).

In [9]:
model_1 = DecisionTreeClassifier(max_depth=2)

model_1.fit(X_train,y_train)

# training data accuracy
y_pred_train = model_1.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = model_1.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9582417582417583
Testing Accuracy: 0.8947368421052632


In [10]:
model_2 = DecisionTreeClassifier(max_depth=3)

model_2.fit(X_train,y_train)

# training data accuracy
y_pred_train = model_2.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = model_2.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9758241758241758
Testing Accuracy: 0.9385964912280702


In [11]:
model_3 = DecisionTreeClassifier(max_depth=4)

model_3.fit(X_train,y_train)

# training data accuracy
y_pred_train = model_3.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = model_3.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9868131868131869
Testing Accuracy: 0.9298245614035088


In [12]:
model_4 = DecisionTreeClassifier(max_depth=5)

model_4.fit(X_train,y_train)

# training data accuracy
y_pred_train = model_4.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = model_4.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9934065934065934
Testing Accuracy: 0.9210526315789473


In [13]:
#grid search
# training data accuracy
y_pred_train = grid_search.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = grid_search.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 1.0
Testing Accuracy: 0.8859649122807017


# Random Forest

## 3. Ensemble Methods: Random Forest Classifier

To improve generalization performance, we implement a Random Forest—an ensemble of uncorrelated Decision Trees. This approach reduces variance through Bootstrap Aggregating (Bagging). We optimize the forest size (`n_estimators`) and feature sampling (`max_features`) to maximize the F1-score and overall predictive reliability.

In [14]:
grid_param={
    "n_estimators" : [100,150,200,300,400],
    "criterion" : ['gini','entropy'],
    "max_features" : ['sqrt','log2'],
}

grid_search_RF = GridSearchCV(
    estimator = RandomForestClassifier(),
    param_grid = grid_param,
    cv = 5
)

grid_search_RF.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_features': ['sqrt', 'log2'],
                         'n_estimators': [100, 150, 200, 300, 400]})

In [15]:
#training data accuracy for RF
y_pred_train = grid_search_RF.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = grid_search_RF.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 1.0
Testing Accuracy: 0.956140350877193
